## 1. Install dependencies

Run this cell once per environment, then restart the kernel if Jupyter requests it.


In [7]:
%pip install -q "psycopg[binary]>=3.2.1" "python-dotenv>=1.0.1" "sentence-transformers>=3.0.1" "pandas>=2.0"


## 2. Configure the environment

In [8]:
import os
import re
from dataclasses import asdict, dataclass
from typing import Literal

import numpy as np
import pandas as pd
import psycopg
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

load_dotenv()

from getpass import getpass
from urllib.parse import quote_plus

db_password = getpass("Database password: ")

DATABASE_URL = (
    "postgresql://"
    f"amazon_user:{quote_plus(db_password)}"
    "@134.122.33.127:5432/amazon_products"
)

print("Database URL configured.")

EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "BAAI/bge-base-en-v1.5")
EXPECTED_EMBEDDING_DIMENSION = 768

print(f"Embedding model: {EMBEDDING_MODEL}")
print("Database URL loaded. Password is intentionally not displayed.")


Database password: ··········
Database URL configured.
Embedding model: BAAI/bge-base-en-v1.5
Database URL loaded. Password is intentionally not displayed.


## 3. Database helpers

In [9]:
def get_connection() -> psycopg.Connection:
    return psycopg.connect(DATABASE_URL)


def fetch_dataframe(sql: str, params: list | tuple | None = None) -> pd.DataFrame:
    with get_connection() as connection:
        with connection.cursor() as cursor:
            cursor.execute(sql, params or ())
            rows = cursor.fetchall()
            columns = [column.name for column in cursor.description]
    return pd.DataFrame(rows, columns=columns)


## 4. Define the structured search request


In [10]:
SortMode = Literal[
    "relevance",
    "price_low_to_high",
    "price_high_to_low",
    "rating",
    "popularity",
]


@dataclass(frozen=True)
class SearchQuery:
    product_description: str
    price_min: float | None = None
    price_max: float | None = None
    min_stars: float | None = None
    min_reviews: int | None = None
    brand: str | None = None
    main_category: str | None = None
    sort_by: SortMode = "relevance"

    def validate(self) -> "SearchQuery":
        if not self.product_description.strip():
            raise ValueError("product_description cannot be empty.")
        if self.price_min is not None and self.price_min < 0:
            raise ValueError("price_min cannot be negative.")
        if self.price_max is not None and self.price_max < 0:
            raise ValueError("price_max cannot be negative.")
        if (
            self.price_min is not None
            and self.price_max is not None
            and self.price_min > self.price_max
        ):
            raise ValueError("price_min cannot be greater than price_max.")
        if self.min_stars is not None and not 0 <= self.min_stars <= 5:
            raise ValueError("min_stars must be between 0 and 5.")
        if self.min_reviews is not None and self.min_reviews < 0:
            raise ValueError("min_reviews cannot be negative.")
        return self


## 5. Construct a query from a common shopping request

In [11]:
# Load fine-tuned Gemma 3 1B IT + LoRA for main-category prediction

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from huggingface_hub import login
login()

BASE_CATEGORY_MODEL = "google/gemma-3-1b-it"
from google.colab import drive
drive.mount('/content/drive')
CATEGORY_ADAPTER_PATH = (
    "/content/drive/MyDrive/shopping_assistant_gemma_lora/"
    "outputs/gemma-3-1b-it-main-category-lora"
)

MAIN_CATEGORY_LABELS = [
    "Arts, Crafts & Party Supplies",
    "Automotive",
    "Baby Products",
    "Beauty & Personal Care",
    "Electronics & Computers",
    "Fashion, Shoes & Luggage",
    "Gift Cards",
    "Health & Household",
    "Home & Kitchen",
    "Industrial & Scientific",
    "Pet Supplies",
    "Smart Home",
    "Sports & Outdoors",
    "Tools & Home Improvement",
    "Toys & Games",
    "Video Games",
]

category_tokenizer = AutoTokenizer.from_pretrained(
    BASE_CATEGORY_MODEL
)

if category_tokenizer.pad_token is None:
    category_tokenizer.pad_token = category_tokenizer.eos_token

category_base_model = AutoModelForCausalLM.from_pretrained(
    BASE_CATEGORY_MODEL,
    device_map="auto",
    dtype=torch.bfloat16,
)

category_model = PeftModel.from_pretrained(
    category_base_model,
    CATEGORY_ADAPTER_PATH,
)

category_model.eval()

print("Fine-tuned Gemma category model loaded.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Fine-tuned Gemma category model loaded.


In [12]:
def clean_category_label(text: str) -> str:
    text = text.strip()
    text = text.split("\n")[0].strip()
    text = text.replace("Answer:", "").strip()
    return text.strip("\"'")


def normalize_category_label(label: str) -> str:
    mapping = {
        "Electronics": "Electronics & Computers",
        "Computers": "Electronics & Computers",
        "Computer": "Electronics & Computers",

        "Fashion": "Fashion, Shoes & Luggage",
        "Fashion Footwear": "Fashion, Shoes & Luggage",
        "Shoes": "Fashion, Shoes & Luggage",
        "Luggage": "Fashion, Shoes & Luggage",

        "Home": "Home & Kitchen",
        "Kitchen": "Home & Kitchen",

        "Toys": "Toys & Games",
        "Games": "Toys & Games",
    }

    label = clean_category_label(label)
    return mapping.get(label, label)


def predict_main_category(user_request: str) -> str:
    labels_text = "\n".join(
        f"- {label}" for label in MAIN_CATEGORY_LABELS
    )

    prompt = f"""You are a product classification assistant.

Classify the user's shopping request into exactly one main product category.

Allowed categories:
{labels_text}

Output only one exact category label from the allowed list.

User request:
{user_request}

Answer:
"""

    inputs = category_tokenizer(
        prompt,
        return_tensors="pt",
    ).to(category_model.device)

    with torch.no_grad():
        outputs = category_model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False,
            pad_token_id=category_tokenizer.eos_token_id,
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    prediction = category_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    )

    prediction = normalize_category_label(prediction)

    if prediction not in MAIN_CATEGORY_LABELS:
        raise ValueError(
            f"Gemma returned an invalid main category: {prediction!r}"
        )

    return prediction

In [13]:

from openai import OpenAI
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Paste your new OpenAI API key: ")

client = OpenAI()

import json
# query constructor
def construct_search_query(user_input):
    schema = {
        "type": "object",
        "properties": {
            "product_description": {
                "type": ["string", "null"],
                "description": "A product search phrase for identifying the best relevant product through embedding search, such as 'wireless gaming mouse'."
            },
            "price_min": {"type": ["number", "null"]},
            "price_max": {"type": ["number", "null"]},
            "min_stars": {"type": ["number", "null"]},
            "min_reviews": {"type": ["integer", "null"]},
            "brand": {"type": ["string", "null"]},
            "sort_by": {
                "type": "string",
                "enum": [
                    "relevance",
                    "price_low_to_high",
                    "price_high_to_low",
                    "rating",
                    "popularity"
                ]
            }
        },
        "required": [
            "product_description",
            "price_min",
            "price_max",
            "min_stars",
            "min_reviews",
            "brand",
            "sort_by"
        ],
        "additionalProperties": False
    }

    response = client.responses.create(
        model="gpt-5.4-nano",
        input=[
            {
                "role": "system",
                "content": (
                    """
You are a shopping-query parser. Convert the user's natural-language request
into structured product-search parameters.

Your output must follow the supplied JSON schema exactly.

GENERAL RULES

1. product_description
- Create a concise, embedding-friendly phrase describing the desired product.
- Include important product attributes such as intended use, recipient,
  size, material, compatibility, style, or key features.
- Do not include price, rating, review count, brand, or sorting instructions
  in product_description when they belong in another field.
- Correct obvious spelling mistakes when the intended product is clear.
- Example:
  "I need a git for my brother around $500"
  becomes:
  "gift for brother"
- However, if "git" likely means "guitar" from the surrounding context,
  use "guitar".

2. Price interpretation
Interpret price language according to the following rules:

- Exact upper limit:
  "under $500", "below $500", "up to $500", "maximum $500"
  -> price_min = null
  -> price_max = 500

- Exact lower limit:
  "over $500", "above $500", "at least $500"
  -> price_min = 500
  -> price_max = null

- Explicit range:
  "between $400 and $600"
  -> price_min = 400
  -> price_max = 600

- Approximate target:
  "around $500", "about $500", "roughly $500", "$500-ish"
  -> treat the amount as the centre of an acceptable range
  -> use a default tolerance of plus or minus 20 percent
  -> price_min = 400
  -> price_max = 600

- Approximate target with a modifier:
  "just under $500"
  -> price_min = 400
  -> price_max = 500

  "a little over $500"
  -> price_min = 500
  -> price_max = 600

- Cheap, affordable, budget, premium, expensive, or luxury without an
  explicit number:
  -> leave price_min and price_max as null
  -> preserve the preference in product_description when useful

- A price described as a budget is normally an upper limit:
  "My budget is $500"
  -> price_min = null
  -> price_max = 500

- A target price is normally a range:
  "I am looking to spend around $500"
  -> price_min = 400
  -> price_max = 600

- Never invent a currency conversion.
- Return only numeric values without currency symbols.

3. Ratings and reviews
- "highly rated", "good ratings", or "well rated"
  -> min_stars = 4.0

- "very highly rated", "top rated", or "excellent ratings"
  -> min_stars = 4.5

- When the user specifies an exact minimum rating, use that value.

- "well reviewed" or "lots of reviews" without a number
  -> do not invent min_reviews
  -> min_reviews = null
  -> use sort_by = "popularity" when popularity is part of the intent

4. Brand
- Extract a brand only when the user explicitly names or clearly requests it.
- Otherwise, brand must be null.
- For exclusions such as "not Apple", do not set brand to Apple because the
  current schema cannot represent excluded brands.

5. Sorting
Choose exactly one sort mode:

- "relevance": default when there is no clear ranking preference
- "price_low_to_high": cheapest, lowest price, budget-first
- "price_high_to_low": most expensive, premium-first
- "rating": best rated, highest rated, top rated
- "popularity": popular, bestselling, most reviewed, widely purchased

A minimum price or maximum price does not automatically change the sort mode.
For example, "a mouse under $50" should still use relevance unless the user
also asks for the cheapest option.

6. Missing information
- Use null for any unspecified optional value.
- Do not invent brands, prices, ratings, review counts, or categories.
- If the request is ambiguous, extract the most defensible interpretation
  rather than adding unsupported details.

EXAMPLES

User: "Find me a wireless gaming mouse around $50 with good ratings"
Output:
{
  "product_description": "wireless gaming mouse",
  "price_min": 40,
  "price_max": 60,
  "min_stars": 4.0,
  "min_reviews": null,
  "brand": null,
  "sort_by": "relevance"
}

User: "I need a premium espresso machine under $1,000"
Output:
{
  "product_description": "premium espresso machine",
  "price_min": null,
  "price_max": 1000,
  "min_stars": null,
  "min_reviews": null,
  "brand": null,
  "sort_by": "relevance"
}

User: "Show me the cheapest highly rated Logitech mechanical keyboard"
Output:
{
  "product_description": "mechanical keyboard",
  "price_min": null,
  "price_max": null,
  "min_stars": 4.0,
  "min_reviews": null,
  "brand": "Logitech",
  "sort_by": "price_low_to_high"
}
"""
                )
            },
            {
                "role": "user",
                "content": user_input
            }
        ],
        text={
            "format": {
                "type": "json_schema",
                "name": "shopping_query",
                "schema": schema,
                "strict": True
            }
        }
    )
    output_json = json.loads(response.output_text)
    return SearchQuery(**output_json).validate()

Paste your new OpenAI API key: ··········


In [14]:
example_request = "I want a wireless gaming mouse under $50 with good reviews, sort by price."
example_query = construct_search_query(example_request)

asdict(example_query)


{'product_description': 'wireless gaming mouse',
 'price_min': None,
 'price_max': 50,
 'min_stars': 4,
 'min_reviews': None,
 'brand': None,
 'main_category': None,
 'sort_by': 'price_low_to_high'}

## 6. Load the query embedding model

In [15]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

def embed_product_description(product_description: str) -> np.ndarray:
    vector = embedding_model.encode(
        product_description,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    vector = np.asarray(vector, dtype=np.float32)

    if vector.shape != (EXPECTED_EMBEDDING_DIMENSION,):
        raise ValueError(
            f"Model returned shape {vector.shape}; "
            f"expected ({EXPECTED_EMBEDDING_DIMENSION},)."
        )
    return vector


def to_pgvector_literal(vector: np.ndarray) -> str:
    return "[" + ",".join(f"{value:.8f}" for value in vector) + "]"


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 7. Semantic product retrieval


In [16]:
SORT_EXPRESSIONS = {
    "relevance": "cosine_distance ASC, reviews DESC",
    "price_low_to_high": "price ASC, cosine_distance ASC",
    "price_high_to_low": "price DESC, cosine_distance ASC",
    "rating": "stars DESC, reviews DESC, cosine_distance ASC",
    "popularity": (
        "bought_in_last_month DESC, reviews DESC, cosine_distance ASC"
    ),
}


def search_products(
    query: SearchQuery,
    *,
    limit: int = 10,
    candidate_limit: int | None = None,
) -> tuple[pd.DataFrame, str, list]:
    query.validate()

    if limit < 1:
        raise ValueError("limit must be at least 1.")

    candidate_limit = candidate_limit or max(200, limit * 20)
    if candidate_limit < limit:
        raise ValueError("candidate_limit must be greater than or equal to limit.")

    query_vector = to_pgvector_literal(
        embed_product_description(query.product_description)
    )

    where_clauses = ["e.model_name = %s"]
    filter_values: list = [EMBEDDING_MODEL]

    if query.price_min is not None:
        where_clauses.append("p.price >= %s")
        filter_values.append(query.price_min)
    if query.price_max is not None:
        where_clauses.append("p.price <= %s")
        filter_values.append(query.price_max)
    if query.min_stars is not None:
        where_clauses.append("p.stars >= %s")
        filter_values.append(query.min_stars)
    if query.min_reviews is not None:
        where_clauses.append("p.reviews >= %s")
        filter_values.append(query.min_reviews)
    if query.brand:
        where_clauses.append("p.title ILIKE %s")
        filter_values.append(f"%{query.brand}%")
    if query.main_category:
        where_clauses.append("p.main_category = %s")
        filter_values.append(query.main_category)

    where_sql = " AND ".join(where_clauses)
    order_sql = SORT_EXPRESSIONS[query.sort_by]

    sql = f"""
        WITH semantic_candidates AS (
            SELECT
                p.asin,
                p.title,
                p.price,
                p.list_price,
                p.stars,
                p.reviews,
                p.is_best_seller,
                p.bought_in_last_month,
                p.main_category,
                p.product_url,
                p.img_url,
                e.title_embedding <=> %s::vector AS cosine_distance
            FROM amazon_product_title_embeddings AS e
            JOIN amazon_products AS p ON p.asin = e.asin
            WHERE p.price != 0 and (
                {where_sql}
                )
            ORDER BY e.title_embedding <=> %s::vector
            LIMIT %s
        )
        SELECT
            asin,
            title,
            price,
            list_price,
            stars,
            reviews,
            is_best_seller,
            bought_in_last_month,
            main_category,
            product_url,
            img_url,
            1.0 - cosine_distance AS similarity
        FROM semantic_candidates
        ORDER BY {order_sql}
        LIMIT %s;
    """

    params = [
        query_vector,
        *filter_values,
        query_vector,
        candidate_limit,
        limit,
    ]
    return fetch_dataframe(sql, params), sql, params


## 8. Run the complete retrieval pipeline

In [17]:
def product_search_pipeline(
    user_request: str,
    *,
    limit: int = 10,
    candidate_limit: int | None = None,
    query_overrides: dict | None = None,
) -> dict:
    # GPT extracts product description, price, rating, brand, and sorting
    parsed_query = construct_search_query(user_request)
    query_data = asdict(parsed_query)

    # Fine-tuned Gemma predicts the main category
    predicted_category = predict_main_category(user_request)
    query_data["main_category"] = predicted_category

    # Optional manual overrides for testing/debugging
    if query_overrides:
        unknown_fields = set(query_overrides) - set(query_data)

        if unknown_fields:
            raise ValueError(
                f"Unknown query fields: {sorted(unknown_fields)}"
            )

        query_data.update(query_overrides)

    search_query = SearchQuery(**query_data).validate()

    products, sql, _params = search_products(
        search_query,
        limit=limit,
        candidate_limit=candidate_limit,
    )

    return {
        "user_request": user_request,
        "query": asdict(search_query),
        "predicted_main_category": predicted_category,
        "sql": sql,
        "results": products,
    }


In [18]:
result = product_search_pipeline(
    "I want a wireless gaming mouse under $50 with good reviews.",
    limit=10,
)

print(result["query"])
print("Gemma category:", result["predicted_main_category"])
display(result["results"])

{'product_description': 'wireless gaming mouse', 'price_min': None, 'price_max': 50, 'min_stars': 4.0, 'min_reviews': None, 'brand': None, 'main_category': 'Video Games', 'sort_by': 'relevance'}
Gemma category: Video Games


,asin,title,price,list_price,stars,reviews,is_best_seller,bought_in_last_month,main_category,product_url,img_url,similarity
0,B09QBTJZ1H,Gaming Mouse (Grey),9.50,0.00,5.00,0,False,0,Video Games,https://www.amazon.com/dp/B09QBTJZ1H,https://m.media-amazon.com/images/I/61slmM8YUo...,0.848831
1,B07M6RVW79,Logitech G MX518 Gaming Mouse,49.99,0.00,4.50,1465,False,0,Video Games,https://www.amazon.com/dp/B07M6RVW79,https://m.media-amazon.com/images/I/51gnOZjDyF...,0.837886
2,B088RMFJNW,Lu737 Pro Gaming Mouse,12.85,0.00,4.00,0,False,0,Video Games,https://www.amazon.com/dp/B088RMFJNW,https://m.media-amazon.com/images/I/61ZrvLwptA...,0.826877
3,B0C71CV62L,Gaming Mouse Computer Mice Mouse Game Mouse 12...,18.00,0.00,4.40,0,False,0,Video Games,https://www.amazon.com/dp/B0C71CV62L,https://m.media-amazon.com/images/I/61bYR0TvME...,0.821321
4,B00HGKOD9G,Mionix AVIOR 7000 Ergonomic Ambidextrous Laser...,19.99,79.99,4.40,0,False,0,Video Games,https://www.amazon.com/dp/B00HGKOD9G,https://m.media-amazon.com/images/I/51ntGFExdz...,0.819501
5,B0932BWJJW,5500DPI USB Wired Gaming Mouse Adjustable 7 Bu...,10.98,0.00,4.20,0,False,0,Video Games,https://www.amazon.com/dp/B0932BWJJW,https://m.media-amazon.com/images/I/51sM3F+IBL...,0.818134
6,B073TZWSFJ,"VicTsing 2.4G Wireless Mouse for PC, Computer",7.49,0.00,4.20,131,False,0,Video Games,https://www.amazon.com/dp/B073TZWSFJ,https://m.media-amazon.com/images/I/51rZ2vYS9f...,0.813080
7,B08FX295MG,Wireless Lightweight Gaming Mouse Honeycomb wi...,15.99,0.00,4.10,0,False,0,Video Games,https://www.amazon.com/dp/B08FX295MG,https://m.media-amazon.com/images/I/71nsx-XosR...,0.811714
8,B08DXD6VDJ,"Lightweight Gaming Mouse,Rechargeable Wireless...",17.99,0.00,4.30,0,False,0,Video Games,https://www.amazon.com/dp/B08DXD6VDJ,https://m.media-amazon.com/images/I/61bz3jIh5y...,0.810996
9,B09W9CZY4Y,Wireless Mouse for Laptop Wireless Gaming Mous...,14.99,0.00,4.30,0,False,0,Video Games,https://www.amazon.com/dp/B09W9CZY4Y,https://m.media-amazon.com/images/I/71wc9isMnl...,0.809000


In [19]:
result = product_search_pipeline(
    "I want to buy a birthday present for my friend who is a soccer fan, my budget is around 100 dollars.",
    limit=10
)

print(result["query"])
display(result["results"])

{'product_description': 'birthday present for a soccer fan', 'price_min': 80, 'price_max': 120, 'min_stars': None, 'min_reviews': None, 'brand': None, 'main_category': 'Toys & Games', 'sort_by': 'relevance'}


,asin,title,price,list_price,stars,reviews,is_best_seller,bought_in_last_month,main_category,product_url,img_url,similarity
0,B09NLYGZF7,ROKR 3D Wooden Puzzles Solar System Model Kit ...,84.99,0.00,4.60,0,False,0,Toys & Games,https://www.amazon.com/dp/B09NLYGZF7,https://m.media-amazon.com/images/I/81NtTxtwZR...,0.611287
1,B09526WJ8V,Topo Gigio Vinyl Doll 20 cm Special Edition,114.90,0.00,4.20,0,False,0,Toys & Games,https://www.amazon.com/dp/B09526WJ8V,https://m.media-amazon.com/images/I/51ogASyRLc...,0.605867
2,B084P6QGNG,"Penny Skateboards Cactus Wanderlust 22""",99.00,0.00,4.60,0,False,0,Toys & Games,https://www.amazon.com/dp/B084P6QGNG,https://m.media-amazon.com/images/I/31kDPOq6wn...,0.600800
3,B0BSRDZRFH,LEGO Friends Sports Center 41744 Building Toy ...,90.00,94.99,5.00,0,False,0,Toys & Games,https://www.amazon.com/dp/B0BSRDZRFH,https://m.media-amazon.com/images/I/81V4TsDKSe...,0.597900
4,B09W56SH5L,"60"" Trampoline for Kids, 5FT Indoor Outdoor Tr...",119.99,139.99,4.50,735,False,100,Toys & Games,https://www.amazon.com/dp/B09W56SH5L,https://m.media-amazon.com/images/I/81vtKpJLoB...,0.597387
5,B07GL6C9QX,Barbie 60th Anniversary Doll,104.95,129.23,4.80,0,False,0,Toys & Games,https://www.amazon.com/dp/B07GL6C9QX,https://m.media-amazon.com/images/I/81BEoavCce...,0.595246
6,B09XTDVSNH,1/14 Scale Rc Box Water Sprayable Fire Truck C...,113.00,0.00,5.00,0,False,0,Toys & Games,https://www.amazon.com/dp/B09XTDVSNH,https://m.media-amazon.com/images/I/61wIa7OhEI...,0.594817
7,B08Z8JN84F,Santa Cruz Rad Dot 9.20in x 33in Cruzer Skateb...,103.95,0.00,2.80,0,False,0,Toys & Games,https://www.amazon.com/dp/B08Z8JN84F,https://m.media-amazon.com/images/I/71nXsadXoh...,0.593370
8,B0BXQ7LGB2,Lego Disney Walt Disney Tribute Camera 43230 D...,99.95,0.00,4.80,86,False,4000,Toys & Games,https://www.amazon.com/dp/B0BXQ7LGB2,https://m.media-amazon.com/images/I/81goEw6klK...,0.591833
9,B0C3HB97T8,14/16 Inch Balance Bike for 3 4 5 6 7 8 Year O...,109.90,0.00,4.00,0,False,0,Toys & Games,https://www.amazon.com/dp/B0C3HB97T8,https://m.media-amazon.com/images/I/71vNwZR432...,0.591117


In [20]:
result = product_search_pipeline(
    "I am looking for a projector that has high resolution and can connect to my phone through screen mirroring, my budget is around 500 dollars.",
    limit=10
)

print(result["query"])
display(result["results"])

{'product_description': 'high-resolution projector with screen mirroring from a phone', 'price_min': 400, 'price_max': 600, 'min_stars': 4.0, 'min_reviews': None, 'brand': None, 'main_category': 'Electronics & Computers', 'sort_by': 'relevance'}


,asin,title,price,list_price,stars,reviews,is_best_seller,bought_in_last_month,main_category,product_url,img_url,similarity
0,B08JGHQNL4,Smart Android Bluetooth Projector 1000 ANSI Lu...,475.00,0.00,4.50,0,False,0,Electronics & Computers,https://www.amazon.com/dp/B08JGHQNL4,https://m.media-amazon.com/images/I/71TEQULwaX...,0.769542
1,B0C77GLG89,"1200 ANSI Lumen Outdoor Projector Auto Focus, ...",499.00,0.00,5.00,4,False,0,Electronics & Computers,https://www.amazon.com/dp/B0C77GLG89,https://m.media-amazon.com/images/I/71J4fMoxzU...,0.759471
2,B09H2KY98L,"Projector with WiFi and Bluetooth, Portable Mi...",599.99,0.00,4.20,205,False,0,Electronics & Computers,https://www.amazon.com/dp/B09H2KY98L,https://m.media-amazon.com/images/I/81hZfLyE3b...,0.758539
3,B0BLRV1MYZ,4K Movie Projector Daylight Viewing 1000ANSI/1...,480.00,0.00,4.00,108,False,0,Electronics & Computers,https://www.amazon.com/dp/B0BLRV1MYZ,https://m.media-amazon.com/images/I/714GC76iox...,0.750247
4,B0CG8R81QW,4K Backyard Movie Projector High Lumens 1100 A...,490.00,0.00,5.00,0,False,0,Electronics & Computers,https://www.amazon.com/dp/B0CG8R81QW,https://m.media-amazon.com/images/I/71hBxR7GDd...,0.749673
5,B081TNBF97,ViewSonic PG707X 4000 Lumens XGA Networkable D...,541.25,0.00,4.50,29,False,0,Electronics & Computers,https://www.amazon.com/dp/B081TNBF97,https://m.media-amazon.com/images/I/61fI-wCG+r...,0.747566
6,B08HCXBRTR,High Brightness Smart Projectors with WiFi and...,443.00,0.00,5.00,0,False,0,Electronics & Computers,https://www.amazon.com/dp/B08HCXBRTR,https://m.media-amazon.com/images/I/71vYPfJW6A...,0.743691
7,B08WR8PMBP,"Miroir M1200S 1080p Smart Projector, 5G WIFI a...",418.69,524.99,4.00,0,False,0,Electronics & Computers,https://www.amazon.com/dp/B08WR8PMBP,https://m.media-amazon.com/images/I/61aynYu8CA...,0.743601
8,B0BM5448BC,LG CineBeam PF510Q Portable Full HD (1920 x 10...,596.99,0.00,5.00,4,False,0,Electronics & Computers,https://www.amazon.com/dp/B0BM5448BC,https://m.media-amazon.com/images/I/61Hq-FjxY-...,0.743544
9,B0CDGR52X3,"Ultra HD Projector for 4K Home Cinema Daytime,...",470.00,0.00,4.40,0,False,0,Electronics & Computers,https://www.amazon.com/dp/B0CDGR52X3,https://m.media-amazon.com/images/I/71sjOleRG5...,0.737866


In [21]:
result = product_search_pipeline(
    "Can u recommend me skin care products for around 100 dollar? My skin is very sensitive, and I am looking for products that can keep it hydrated",
    limit=10
)
print(result["query"])
display(result["results"])

{'product_description': 'skincare products for very sensitive skin focused on hydration and moisturizing (gentle, fragrance-free)', 'price_min': 80, 'price_max': 120, 'min_stars': None, 'min_reviews': None, 'brand': None, 'main_category': 'Beauty & Personal Care', 'sort_by': 'relevance'}


,asin,title,price,list_price,stars,reviews,is_best_seller,bought_in_last_month,main_category,product_url,img_url,similarity
0,B000IOLCWI,"Hydra-Cool Serum, Refreshing and Hydrating Ski...",96.00,0.00,4.60,0,False,500,Beauty & Personal Care,https://www.amazon.com/dp/B000IOLCWI,https://m.media-amazon.com/images/I/61pzJpxam5...,0.698832
1,B003TY8VPA,"Revision Skincare Hydrating Serum, with hyalur...",100.00,0.00,4.60,0,False,300,Beauty & Personal Care,https://www.amazon.com/dp/B003TY8VPA,https://m.media-amazon.com/images/I/41xj5SDAPd...,0.691496
2,B085GLWN8G,Hydro-Drops Face Serum Formulated with Vitamin...,105.00,0.00,4.60,0,False,400,Beauty & Personal Care,https://www.amazon.com/dp/B085GLWN8G,https://m.media-amazon.com/images/I/61KapJKutM...,0.687456
3,B0013QOIGC,Vitamin E Creme Swiss Collagen Complex Moistur...,98.00,0.00,4.60,0,False,300,Beauty & Personal Care,https://www.amazon.com/dp/B0013QOIGC,https://m.media-amazon.com/images/I/71sg11QYMo...,0.676084
4,B072899CXM,Biotene Dry Mouth Moisturizing Spray Gentle Mi...,82.85,0.00,4.50,0,False,0,Beauty & Personal Care,https://www.amazon.com/dp/B072899CXM,https://m.media-amazon.com/images/I/61bqu904PR...,0.673112
5,B07H1HNCCF,"IMAGE Skincare, AGELESS Total Pure Hyaluronic ...",80.00,0.00,4.60,0,False,300,Beauty & Personal Care,https://www.amazon.com/dp/B07H1HNCCF,https://m.media-amazon.com/images/I/41sUgxbR-k...,0.667571
6,B006JXWVF4,CHANEL Chance Eau Tendre Voile Hydratant Moist...,95.69,0.00,4.60,0,False,50,Beauty & Personal Care,https://www.amazon.com/dp/B006JXWVF4,https://m.media-amazon.com/images/I/61REAdQEcz...,0.665938
7,B0BYH4P291,Anti-Aging Daily Skincare System with Youth Ac...,89.00,0.00,4.20,0,False,600,Beauty & Personal Care,https://www.amazon.com/dp/B0BYH4P291,https://m.media-amazon.com/images/I/61Nvvvg4W9...,0.665512
8,B0017OLMVE,Estee Lauder Idealist Pore Minimizing Skin Ref...,102.00,0.00,4.60,736,False,300,Beauty & Personal Care,https://www.amazon.com/dp/B0017OLMVE,https://m.media-amazon.com/images/I/41quDYl87U...,0.665228
9,B097T15823,TASALON Facial Steamer on Wheels - Face Steame...,99.99,0.00,4.10,0,False,100,Beauty & Personal Care,https://www.amazon.com/dp/B097T15823,https://m.media-amazon.com/images/I/618IM9seez...,0.663771


## 9. Product Recommendation Generator

In [22]:
## Helper Functions

PRODUCT_FIELDS = [
    "asin", "title", "price", "stars", "reviews",
    "is_best_seller", "bought_in_last_month", "main_category",
    "product_url", "similarity",
]

RECOMMENDATION_ITEM_SCHEMA = {
    "type": "object",
    "properties": {
        "rank": {"type": "integer"},
        "asin": {"type": "string"},
        "reason": {"type": "string"},
    },
    "required": ["rank", "asin", "reason"],
    "additionalProperties": False,
}

RECOMMENDATION_MODEL = 'gpt-5.4-nano'

def _json_safe(value):
    if value is None or pd.isna(value):
        return None
    if isinstance(value, np.generic):
        return value.item()
    return value

def _prepare_products(results: pd.DataFrame, count: int) -> list[dict]:
    products = []
    for rank, (_, row) in enumerate(results.head(count).iterrows(), start=1):
        product = {field: _json_safe(row.get(field)) for field in PRODUCT_FIELDS}
        product["rank"] = rank
        product["asin"] = str(product["asin"])
        products.append(product)
    return products

def _format_product_message(product: dict) -> str:
    title = product["title"]
    url = product["url"]
    heading = f"[{title}]({url})" if url else title
    facts = []
    if product["price"] is not None:
        facts.append(f"Price: ${float(product['price']):,.2f}")
    if product["stars"] is not None:
        facts.append(f"Rating: {float(product['stars']):g}/5")
    if product["reviews"] is not None:
        facts.append(f"{int(product['reviews']):,} reviews")
    fact_text = f" ({' · '.join(facts)})" if facts else ""
    return f"{product['rank']}. **{heading}**{fact_text}\n\n   {product['reason']}"

def _build_message(user_request: str, products: list[dict]) -> str:
    intro = f"Here are the top {len(products)} results for your request, in the order returned by the search:"
    return intro + "\n\n" + "\n\n".join(_format_product_message(product) for product in products)

In [23]:
def generate_product_recommendations(search_output: dict, top_x: int = 3) -> dict:
    if isinstance(top_x, bool) or not isinstance(top_x, int) or top_x < 1:
        raise ValueError("top_x must be an integer of at least 1.")
    if not isinstance(search_output, dict):
        raise TypeError("search_output must be the dictionary returned by product_search_pipeline.")
    if "user_request" not in search_output or "results" not in search_output:
        raise ValueError("search_output must contain 'user_request' and 'results'.")
    results = search_output["results"]
    if not isinstance(results, pd.DataFrame):
        raise TypeError("search_output['results'] must be a pandas DataFrame.")
    if "asin" not in results.columns:
        raise ValueError("search_output['results'] must contain an 'asin' column.")
    if results.empty:
        return {
            "selected_products": [],
            "message": "I couldn't find any matching products. Try broadening the product description or relaxing a price, rating, brand, or review filter.",
        }

    count = min(top_x, len(results))
    source_products = _prepare_products(results, count)
    expected_asins = [product["asin"] for product in source_products]
    schema = {
        "type": "object",
        "properties": {
            "recommendations": {
                "type": "array",
                "items": RECOMMENDATION_ITEM_SCHEMA,
                "minItems": count,
                "maxItems": count,
            }
        },
        "required": ["recommendations"],
        "additionalProperties": False,
    }
    model_input = {
        "user_request": search_output["user_request"],
        "parsed_query": search_output.get("query", {}),
        "products_in_required_order": source_products,
    }
    response = client.responses.create(
        model=RECOMMENDATION_MODEL,
        input=[
            {
                "role": "system",
                "content": (
"""
Write one concise, friendly, and shopping-oriented recommendation reason for every supplied product.

Tailor each reason to the user’s stated request, preferences, budget, intended recipient, occasion, and constraints. Clearly explain why the product may be a good fit for this specific user rather than giving a generic product summary.

Return every supplied product exactly once and in its original rank order. Never select, omit, replace, duplicate, or rerank products.

Use only facts explicitly provided in the user request and product data. Do not infer unlisted features, compatibility, quality, materials, sizing, use cases, recipient preferences, or product capabilities from the title alone. Do not fill in missing values, convert currencies, or claim personal experience.

When the available facts do not confirm an important requested feature, acknowledge the limitation naturally and suggest that the user verify it rather than presenting it as certain.

Do not repeat the product’s price, rating, review count, rank, URL, or other metadata that the application already displays. Focus each reason on the product’s relevance to the user’s needs and avoid repeating the same justification across products.

Keep each recommendation specific, natural, and concise—ideally one or two sentences.

"""
                ),
            },
            {"role": "user", "content": json.dumps(model_input, ensure_ascii=False, default=float, indent=5)},
        ],
        text={"format": {"type": "json_schema", "name": "product_recommendations", "schema": schema, "strict": True}},
    )
    generated = json.loads(response.output_text)["recommendations"]
    actual_asins = [str(item["asin"]) for item in generated]
    actual_ranks = [item["rank"] for item in generated]
    if actual_asins != expected_asins or actual_ranks != list(range(1, count + 1)):
        raise ValueError("The model changed product identity or retrieval order; recommendation output was rejected.")

    selected_products = []
    for source, generated_item in zip(source_products, generated):
        selected_products.append({
            "rank": source["rank"],
            "asin": source["asin"],
            "title": source["title"],
            "price": source["price"],
            "stars": source["stars"],
            "reviews": source["reviews"],
            "url": source["product_url"],
            "reason": generated_item["reason"].strip(),
        })
    return {
        "selected_products": selected_products,
        "message": _build_message(search_output["user_request"], selected_products),
    }

def personalized_product_search(user_request: str, top_x: int = 3, candidate_limit: int | None = None) -> dict:
    if isinstance(top_x, bool) or not isinstance(top_x, int) or top_x < 1:
        raise ValueError("top_x must be an integer of at least 1.")
    search_output = product_search_pipeline(user_request, limit=top_x, candidate_limit=candidate_limit)
    recommendations = generate_product_recommendations(search_output, top_x=top_x)
    return {"search": search_output, **recommendations}

In [24]:
from IPython.display import display, Markdown
existing_search_output = product_search_pipeline(
    "I need a birthday present for a soccer fan around $100.",
    limit=10,
)
existing_recommendations = generate_product_recommendations(existing_search_output, top_x=3)
#display(existing_recommendations["selected_products"])
display(Markdown(existing_recommendations["message"]))

Here are the top 3 results for your request, in the order returned by the search:

1. **[ROKR 3D Wooden Puzzles Solar System Model Kit for Adults to Build Mechanical Orrery Desk Display Gift for Birthday](https://www.amazon.com/dp/B09NLYGZF7)** (Price: $84.99 · Rating: 4.6/5 · 0 reviews)

   This 3D wooden puzzle solar system model kit is a great birthday gift option around your budget—especially if the soccer fan also enjoys building and desk-display projects. It’s positioned as an adult-build kit, so it may feel like a more personal, hands-on present.

2. **[Topo Gigio Vinyl Doll 20 cm Special Edition](https://www.amazon.com/dp/B09526WJ8V)** (Price: $114.90 · Rating: 4.2/5 · 0 reviews)

   If you want a straightforward birthday gift that still fits your ~$100 budget, this Special Edition Topo Gigio vinyl doll could be a fun pick for a soccer fan who likes collectible characters. Since it’s a specific themed collectible, it may feel more “giftable” than a generic item.

3. **[Penny Skateboards Cactus Wanderlust 22"](https://www.amazon.com/dp/B084P6QGNG)** (Price: $99.00 · Rating: 4.6/5 · 0 reviews)

   A skateboard is a sporty, outdoorsy birthday present that fits your around-$100 budget, and it matches the active vibe many soccer fans enjoy. The 22" “Cactus Wanderlust” style also gives it a distinctive, gift-worthy look—just double-check any preferences on board size.

In [25]:
existing_search_output = product_search_pipeline(
    "Can u recommend me skin care products for around 100 dollar? My skin is very sensitive, and I am looking for products that can keep it hydrated",
    limit=10,
)
existing_recommendations = generate_product_recommendations(existing_search_output, top_x=5)
#display(existing_recommendations["selected_products"])
display(Markdown(existing_recommendations["message"]))

Here are the top 5 results for your request, in the order returned by the search:

1. **[Hydro-Drops Face Serum Formulated with Vitamin B3 and Hibiscus Oil, Hypoallergenic and Dermatologist Tested–Anti-Aging Hydrating Moisture Serum Suitable for all Skin Types, 1-FL Oz. Pack-of-1](https://www.amazon.com/dp/B085GLWN8G)** (Price: $105.00 · Rating: 4.6/5 · 0 reviews)

   A hypoallergenic, dermatologist-tested hydrating face serum that targets moisture with Vitamin B3 and hibiscus oil—nice if you have very sensitive skin and want help keeping it hydrated.

2. **[Hydra-Cool Serum, Refreshing and Hydrating Skin Face Serum, Anti-Blemish, Anti-Redness](https://www.amazon.com/dp/B000IOLCWI)** (Price: $96.00 · Rating: 4.6/5 · 0 reviews)

   This refreshing, hydrating face serum is specifically positioned for anti-redness and anti-blemish needs, which can be helpful for sensitive skin while you’re aiming for better hydration.

3. **[Anti-Aging Daily Skincare System with Youth Activating Serum](https://www.amazon.com/dp/B0BYH4P291)** (Price: $89.00 · Rating: 4.2/5 · 0 reviews)

   An anti-aging daily skincare system with a youth-activating serum option that stays within your budget—worth a look if you want hydration support as part of an everyday routine.

4. **[Revision Skincare Hydrating Serum, with hyaluronic acid and fruit extracts, provides short and long term moisturization, reduce fine lines and wrinkles, keeps skin hydrated, oil free moisture, 1 Fl oz](https://www.amazon.com/dp/B003TY8VPA)** (Price: $100.00 · Rating: 4.6/5 · 0 reviews)

   A hydrating serum that’s described as providing short- and long-term moisturization and keeping skin hydrated, and it’s noted as oil-free moisture—good for sensitive skin looking for hydration without heaviness.

5. **[Vitamin E Creme Swiss Collagen Complex Moisturizing Creme for Dry and Sensitive Skin 16 oz](https://www.amazon.com/dp/B0013QOIGC)** (Price: $98.00 · Rating: 4.6/5 · 0 reviews)

   A moisturizing creme made for dry and sensitive skin, with a gentle-sounding moisture focus that may help you maintain hydration within your ~$100 budget—just double-check how it fits your routine.

In [26]:
existing_search_output = product_search_pipeline(
    "What keyboard would you recommend for a new gamer to buy for a budget under 50 dollars",
    limit=10,
)
existing_recommendations = generate_product_recommendations(existing_search_output, top_x=5)
#display(existing_recommendations["selected_products"])
display(Markdown(existing_recommendations["message"]))

Here are the top 5 results for your request, in the order returned by the search:

1. **[One Hand RGB Gaming Keyboard,USB Wired Rainbow Letters Glow Single Hand Keyboard with Wrist Rest Support Multimedia Keys, Backlit Ergonomic Mechanical Feeling Keyboard for Game](https://www.amazon.com/dp/B08B1GS1ZG)** (Price: $17.99 · Rating: 4.3/5 · 3,290 reviews)

   This one-hand wired RGB keyboard is a budget-friendly option under $50 for a new gamer, and the included wrist rest support and multimedia keys can help make gaming basics more comfortable.

2. **[Gaming Keyboard for Girl, 60 Percent Keyboard Color Cute Keyboard with RGB, Wired Mechanical Keyboard for Gaming Office Apricot](https://www.amazon.com/dp/B0BBW138VB)** (Price: $26.22 · Rating: 4.2/5 · 0 reviews)

   If you want something under $50 with a cute, gamer-style look, this 60% wired RGB keyboard is an affordable choice—just double-check the key layout and specs since it’s aimed at “for girl” buyers in the title.

3. **[One Hand RGB Gaming Keyboard and Backlit Mouse Combo,USB Wired Rainbow Letters Glow Single Hand Mechanical Feeling Keyboard with Wrist Rest Support, Gaming Keyboard Set for Game](https://www.amazon.com/dp/B07YFNW54T)** (Price: $23.99 · Rating: 4.3/5 · 3,290 reviews)

   For a simple starter setup on a budget, this single-hand keyboard bundled with a backlit mouse keeps you under $50 and adds wrist rest support for more comfortable sessions.

4. **[HORI EDGE 201 Mechanical Gaming Keyboard (EGU-201)](https://www.amazon.com/dp/B00ZXAXR0M)** (Price: $19.95 · Rating: 3.9/5 · 0 reviews)

   The HORI EDGE 201 mechanical gaming keyboard is a low-cost pick under $50 with a well-known gaming brand; it’s a good way to get a dedicated mech-style option—verify details like layout and switch type on the listing.

5. **[75% Mechanical Gaming Keyboard with Red Switch, RGB Backlit Keyboard, 87 Keys Compact TKL Wired Computer Keyboard for Laptop PC Gamer Xbox PS (Grey/ 87 Red Switch)](https://www.amazon.com/dp/B0C4JKSFTV)** (Price: $34.99 · Rating: 4.6/5 · 0 reviews)

   With a 75% compact size, red switch, and RGB backlighting for under $50, this is a strong option for a new gamer who wants a more space-saving board—just confirm it matches your preferred key layout before buying.

In [27]:
existing_search_output = product_search_pipeline(
    "I am looking for a projector that has high resolution and can connect to my phone through screen mirroring, my budget is around 500 dollars, and please give me products with ratings at least 4.5",
    limit=10,
)
existing_recommendations = generate_product_recommendations(existing_search_output, top_x=5)
display(Markdown(existing_recommendations["message"]))

Here are the top 5 results for your request, in the order returned by the search:

1. **[Smart Android Bluetooth Projector 1000 ANSI Lumens, 4K Support Wireless 5G WiFi Mirroring Native 1080P HD Projector 200" Home Theater Outdoor Movie Gaming with Digital Zoom 4D Keystone HDMI USB Apps](https://www.amazon.com/dp/B08JGHQNL4)** (Price: $475.00 · Rating: 4.5/5 · 0 reviews)

   With 4K support and native 1080P HD, this budget-friendly projector also includes wireless 5G WiFi mirroring, which should make it a good match for screen mirroring from your phone.

2. **[1200 ANSI Lumen Outdoor Projector Auto Focus, Auto Keystone 4K Projector with 5G WiFi 6 Bluetooth, Built-in Android TV WLAN, Daytime Home Theater Proyector 300'' Display for Cell Phones PC DVD PPT](https://www.amazon.com/dp/B0C77GLG89)** (Price: $499.00 · Rating: 5/5 · 4 reviews)

   This option is rated 5.0 and is described as offering 4K with 5G WiFi and built-in Android TV, making it a strong candidate if you want phone mirroring plus a higher-resolution projector within your budget.

3. **[LG CineBeam PF510Q Portable Full HD (1920 x 1080) LED Smart Projector, Airplay 2 and Screen Share support, Bluetooth Audio Dual Out](https://www.amazon.com/dp/B0BM5448BC)** (Price: $596.99 · Rating: 5/5 · 4 reviews)

   At a 5.0 rating, it’s a Full HD (1920 x 1080) portable smart projector and explicitly supports Airplay 2 and Screen Share, which aligns well with connecting from your phone via mirroring.

4. **[4K Backyard Movie Projector High Lumens 1100 ANSI Auto Keystone WiFi 6 Digital LED LCD Smart 4K Projector 5G WiFi Bluetooth Android Apps HDMI USB RJ45 Port 1080P Full HD Native Wireless Mirroring](https://www.amazon.com/dp/B0CG8R81QW)** (Price: $490.00 · Rating: 5/5 · 0 reviews)

   It’s listed as a 4K smart projector with wireless mirroring and Android apps, and it also sits in your ~$500 range—worth checking if your priority is high-resolution plus phone screen mirroring.

5. **[ViewSonic PG707X 4000 Lumens XGA Networkable DLP Projector with HDMI 1.3x Optical Zoom and Low Input Lag for Home and Corporate Settings](https://www.amazon.com/dp/B081TNBF97)** (Price: $541.25 · Rating: 4.5/5 · 29 reviews)

   This ViewSonic projector is rated 4.5 and offers high brightness (4000 lumens) and HDMI connectivity, which can be helpful if you’ll be routing your phone’s display through an HDMI-based mirroring setup—just confirm mirroring method details before buying.

In [28]:
existing_search_output = product_search_pipeline(
    "I am looking for a projector that has high resolution and can connect to my phone through screen mirroring, my budget is around 500 dollars, filter for products with at least 10 reviews",
    limit=10,
)
existing_recommendations = generate_product_recommendations(existing_search_output, top_x=5)
display(Markdown(existing_recommendations["message"]))

Here are the top 5 results for your request, in the order returned by the search:

1. **[4K Movie Projector Daylight Viewing 1000ANSI/13000lm Smart App Streaming Bluetooth WiFi Outdoor Projector with Android TV 2G+16G LAN HDMI USB, Ultra HD Wireless Home Projectors for Gaming Presentation](https://www.amazon.com/dp/B0BLRV1MYZ)** (Price: $480.00 · Rating: 4/5 · 108 reviews)

   This 4K projector in your budget has strong social proof (108 reviews) and is built for wireless streaming with WiFi/Android TV plus Bluetooth, which should make phone-to-projector mirroring easier—just confirm screen mirroring is explicitly supported for your specific phone model.

2. **[Projector with WiFi and Bluetooth, Portable Mini Projector 4K Outdoor Movie Projector, Home Smart Phone Projector, 1080P Projector HDMI, VGA, TV Stick, USB, SD Card Supported](https://www.amazon.com/dp/B09H2KY98L)** (Price: $599.99 · Rating: 4.2/5 · 205 reviews)

   With 205 reviews and a price right at the top of your range, this portable 4K-capable projector includes WiFi/Bluetooth and multiple input options, making it a practical choice if you want to connect from a smartphone via screen mirroring—verify the mirroring method is supported on your phone.

3. **[ViewSonic PG707X 4000 Lumens XGA Networkable DLP Projector with HDMI 1.3x Optical Zoom and Low Input Lag for Home and Corporate Settings](https://www.amazon.com/dp/B081TNBF97)** (Price: $541.25 · Rating: 4.5/5 · 29 reviews)

   If you want a higher-end option with low input lag for home or work use, the ViewSonic has 29 reviews and strong brightness specs; it includes networking and HDMI, so you may be able to mirror from your phone depending on what screen-mirroring app/connection method you plan to use—check compatibility.

4. **[4K Smart Ultra HD Projector 5G WiFi Bluetooth, 1100 ANSI/14300 Lumen Movie Projector Daytime 300” Display, RJ45 LAN Android Projector with Apps Keystone, Zoom, Outdoor, Home Entertainment and Work](https://www.amazon.com/dp/B0BRNBSB35)** (Price: $490.00 · Rating: 4/5 · 27 reviews)

   This model is positioned as a 4K smart projector with WiFi and Bluetooth, and it has 27 reviews, aligning with your desire for high resolution and wireless smartphone connection—please confirm screen mirroring support matches your phone.

5. **[BenQ GV30 LED Portable Ceiling Projector with 135˚ Rotating Angle Projection | Extra Bass Bluetooth Speaker | Android TV 10 |Auto Focus & Vertical Keystone | WiFi |Chromecast & AirPlay | HDMI | USB-C](https://www.amazon.com/dp/B09BTKPB1N)** (Price: $429.00 · Rating: 4.5/5 · 142 reviews)

   BenQ’s portable projector pairs well with mobile streaming because it includes Android TV and Chromecast/AirPlay, and it has 142 reviews; that combination often aligns closely with phone screen mirroring, so it’s worth checking your phone’s AirPlay/Chromecast support.